# 第62章 交互箱线图（px.box）

用交互箱线图比较分布摘要，并通过Hover查看具体异常值。

## 学习目标

本章围绕一种明确的图表结构展开，先看最小可用示例，再加入分组、注释或交互细节。学习重点不是“把图画出来”，而是让图表服务于一个可回答的问题。


## 适用场景

比较类别组的中位数、离散程度和潜在异常。

## 数据结构

一列分类和一列连续数值。

## 本章练习任务

运行基础图表后，完成以下任务：

1. 将 points="outliers" 改为 points="all" 或 points=False，观察显示点数量的差异
2. 添加 notched=True 参数，对比缺口箱线图与标准箱线图的中位数比较效果
3. 修改 hover_data 增加补充字段，说明悬浮信息对异常点溯源的作用


## 0. 准备可复现数据

先完成导入和数据准备，后续单元格只负责一种图表或一种分析动作。


In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

funnel = pd.DataFrame({
    "stage": ["访问", "查看商品", "加入购物车", "提交订单", "支付成功"],
    "users": [12000, 7200, 3100, 1850, 1420],
})
timeline = pd.DataFrame({
    "task": ["数据准备", "探索分析", "图表制作", "报告复核"],
    "start": pd.to_datetime(["2026-03-01", "2026-03-04", "2026-03-08", "2026-03-12"]),
    "finish": pd.to_datetime(["2026-03-04", "2026-03-09", "2026-03-13", "2026-03-15"]),
    "owner": ["数据", "分析", "分析", "负责人"],
})
from js import window
base_url = window.location.origin
diamonds = pd.read_csv(f"{base_url}/datasets/diamonds.csv")
orders_full = diamonds.assign(
    date=pd.Timestamp("2026-01-01"), category=diamonds["cut"], region=diamonds["clarity"],
    channel=diamonds["color"], order_value=diamonds["price"], items=diamonds["carat"],
    sales=diamonds["price"], month="公开样本",
)
orders = orders_full.sample(5_000, random_state=55).copy()
flights = pd.read_csv(f"{base_url}/datasets/flights.csv")
monthly = flights.query("year == 1960").rename(columns={"passengers": "sales"}).copy()
monthly["orders"] = monthly["sales"]
monthly["profit"] = monthly["sales"].rolling(3, min_periods=1).mean()
regional = orders_full.groupby(["region", "channel"], as_index=False)["sales"].sum()
hierarchy = diamonds.groupby(["cut", "color"], as_index=False)["price"].sum().rename(
    columns={"cut": "department", "color": "category", "price": "sales"}
)
gapminder = pd.read_csv(f"{base_url}/datasets/gapminder.csv")
countries = gapminder.query("year == 2007").assign(
    country=lambda frame: frame["country"], market=lambda frame: frame["country"],
    sales=lambda frame: frame["gdpPercap"], growth=lambda frame: frame["lifeExp"]
)
print(f"Diamonds：{len(diamonds):,} 行；Flights：{len(flights):,} 行；Gapminder：{len(gapminder):,} 行")


## 1. 基础图表

先保留必要的编码：位置、颜色或大小。图表标题、坐标轴和单位应能让读者脱离代码理解结果。


In [ ]:
fig = px.box(orders, x="category", y="order_value", color="category", points="outliers", title="品类客单价箱线图")
fig.update_layout(xaxis_title="品类", yaxis_title="客单价（元）", showlegend=False)
fig.show()


## 2. 进阶变体

在基础图表可读的前提下增加分组、布局、注释或交互。新增编码必须服务于一个明确问题。


In [ ]:
fig = px.box(orders, x="region", y="order_value", color="channel", points="suspectedoutliers", notched=True, title="区域渠道客单价")
fig.update_layout(xaxis_title="区域", yaxis_title="客单价（元）", legend_title="渠道")
fig.show()


## 3. 参数说明

- points：显示点
- notched：缺口
- color：分组
- hover_data：补充信息


## 4. 结果解读

读取箱体、中位数和须；悬浮异常点确认其所属类别和字段。


## 常见误区

- 点全部显示造成拥挤
- 把异常点自动判为错误
- 不同组样本量不可见


## 综合练习

请使用同一份数据完成下面任务，并说明你选择该图表的原因。完成后补充：图表回答了什么问题、最重要的视觉信号是什么、还有哪些信息无法从图中得出。


In [ ]:
fig = px.box(orders, x="category", y="items", color="region", points="all", title="品类购买件数分布")
fig.update_traces(jitter=0.25, pointpos=0)
fig.update_layout(xaxis_title="品类", yaxis_title="购买件数", legend_title="区域")
fig.show()


## 本章小结

用交互箱线图比较分布摘要，并通过Hover查看具体异常值。


### 你已经掌握

- 判断交互箱线图（px.box）的适用场景
- 准备与图表匹配的数据结构
- 从基础图表扩展到分组、注释或交互变体
- 按照业务问题解读图表并说明结论边界


### 图表选择速查

| 选择要点 | 本章说明 |
| --- | --- |
| 适用场景 | 比较类别组的中位数、离散程度和潜在异常。 |
| 数据结构 | 一列分类和一列连续数值。 |
| 结果解读 | 读取箱体、中位数和须；悬浮异常点确认其所属类别和字段。 |


### 关键参数

| 参数 | 作用 |
| --- | --- |
| `points` | 显示点 |
| `notched` | 缺口 |
| `color` | 分组 |
| `hover_data` | 补充信息 |


### 需要注意

- 点全部显示造成拥挤
- 把异常点自动判为错误
- 不同组样本量不可见


### 完成检查

- [ ] 能判断什么问题适合使用交互箱线图（px.box）
- [ ] 能准备符合要求的数据结构
- [ ] 能独立完成基础图表和一个进阶变体
- [ ] 能调整关键参数并解释视觉变化
- [ ] 能根据图表写出有边界的数据结论


### 下一步推荐

把同一图表迁移到另一份数据，先保留同样的编码，再只改变一个维度。比较迁移前后的可读性，并说明哪些结论仍然成立。
